In [1]:
!pip install pyspark -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_energy_usage.csv to cleaned_energy_usage.csv


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SmartHomeETL").getOrCreate()

df = spark.read.csv("cleaned_energy_usage.csv", header=True, inferSchema=True)
df.show(5)

+-------------------+---------+-------+----------+
|          timestamp|device_id|room_id|energy_kwh|
+-------------------+---------+-------+----------+
|2025-07-03 19:30:00|       10|      2|     0.021|
|2025-01-24 21:00:00|        4|      2|      0.05|
|2025-03-18 23:00:00|        8|      5|       0.0|
|2025-01-09 21:30:00|        3|      2|     0.028|
|2025-05-29 23:30:00|        3|      2|     0.036|
+-------------------+---------+-------+----------+
only showing top 5 rows


In [4]:
daily_summary = (
    df.withColumn("date", F.to_date("timestamp"))
      .groupBy("device_id", "room_id", "date")
      .agg(F.sum("energy_kwh").alias("total_kwh"),
           F.avg("energy_kwh").alias("avg_kwh"))
      .orderBy("date", "device_id")
)

weekly_summary = (
    df.withColumn("week_start", F.to_date(F.date_trunc("week", "timestamp")))
      .groupBy("device_id", "room_id", "week_start")
      .agg(F.sum("energy_kwh").alias("total_kwh"),
           F.avg("energy_kwh").alias("avg_kwh"))
      .orderBy("week_start", "device_id")
)

daily_summary.show(10)
weekly_summary.show(10)

+---------+-------+----------+------------------+--------------------+
|device_id|room_id|      date|         total_kwh|             avg_kwh|
+---------+-------+----------+------------------+--------------------+
|        1|      1|2025-01-01|11.809000000000003|  0.2567173913043479|
|        2|      1|2025-01-01|1.0729999999999993|0.023326086956521722|
|        3|      2|2025-01-01|0.7610000000000002|0.016191489361702133|
|        4|      2|2025-01-01|0.8870000000000003|0.018872340425531924|
|        5|      3|2025-01-01|2.6640000000000006| 0.05791304347826088|
|        6|      3|2025-01-01|             1.897| 0.04215555555555556|
|        7|      4|2025-01-01|0.7849999999999999|0.017444444444444443|
|        8|      5|2025-01-01|             5.838|            0.121625|
|        9|      1|2025-01-01|0.2810000000000002|0.005978723404255324|
|       10|      2|2025-01-01|             0.267|0.005804347826086957|
+---------+-------+----------+------------------+--------------------+
only s

In [5]:
daily_summary.toPandas().to_csv("daily_summary.csv", index=False)
weekly_summary.toPandas().to_csv("weekly_summary.csv", index=False)

In [6]:
files.download("daily_summary.csv")
files.download("weekly_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>